In [ ]:
import numpy as np
import pandas as pd
from numba import njit, prange

COURSES = 100_000
BATCH = 2000
SURFACE_POINTS = 4096
INNER_POINTS = 1024
SEED = 42
SIGMA_AXIS = 3.0 / np.sqrt(3.0 - 8.0 / np.pi)


def sphere(n, radius, phase=0.0):
    k = np.arange(n)
    z = 1 - 2 * (k + 0.5) / n
    phi = k * np.pi * (3 - np.sqrt(5)) + phase
    return radius * np.column_stack((
        np.sqrt(1-z*z) * np.cos(phi),
        np.sqrt(1-z*z) * np.sin(phi),
        z
    ))


points = np.vstack((
    sphere(SURFACE_POINTS, 20),
    sphere(INNER_POINTS, 16, 0.5),
    np.zeros((1, 3))
))


@njit
def passes(excess, margin):
    loss = 0.0
    for e in excess:
        x = e - margin
        if x > 10:
            loss += 1
        elif x > 0:
            loss += x / 100
        if loss > 0.1 * len(excess) + 1e-12:
            return False
    return True


@njit
def required_margin(shifts, positions):
    n = len(shifts)
    excess = np.empty(n)
    worst = 0.0

    for j in range(len(positions) + n):
        if j < n:
            v = shifts[j]
            r = np.sqrt(v[0]**2 + v[1]**2 + v[2]**2)
            if r == 0:
                continue
            x, y, z = 20*v[0]/r, 20*v[1]/r, 20*v[2]/r
        else:
            x, y, z = positions[j-n]

        for i in range(n):
            dx = x + shifts[i, 0]
            dy = y + shifts[i, 1]
            dz = z + shifts[i, 2]
            excess[i] = np.sqrt(dx*dx + dy*dy + dz*dz) - 20

        if passes(excess, worst):
            continue

        lo = worst
        hi = max(worst, np.max(excess))

        while hi - lo > 0.002:
            mid = (lo + hi) / 2
            if passes(excess, mid):
                hi = mid
            else:
                lo = mid

        worst = hi

    return worst


@njit(parallel=True)
def required_batch(shifts, positions):
    margins = np.empty(len(shifts))
    for j in prange(len(shifts)):
        margins[j] = required_margin(shifts[j], positions)
    return margins


rng = np.random.default_rng(SEED)
critical = {}

for n in (3, 30):
    collected = []

    for start in range(0, COURSES, BATCH):
        size = min(BATCH, COURSES - start)
        shifts = rng.normal(0, SIGMA_AXIS, (size, n, 3))

        if n == 3:
            margins = np.maximum(
                0, np.linalg.norm(shifts, axis=2).max(axis=1) - 10
            )
        else:
            margins = required_batch(shifts, points)

        collected.append(margins)

    critical[n] = np.sort(np.concatenate(collected))


def confidence(n, margin):
    a = critical[n]
    return 100 * np.searchsorted(a, margin, side='right') / len(a)


steps = []
results = []

for n in (3, 30):
    lo, hi = 0.0, 10.0

    for iteration in range(1, 12):
        mid = (lo + hi) / 2
        p = confidence(n, mid)
        steps.append((n, iteration, round(mid, 4), round(p, 3)))

        if p >= 95:
            hi = mid
        else:
            lo = mid

        if hi - lo <= 0.01:
            break

    m95 = critical[n][int(np.ceil(0.95 * COURSES)) - 1]
    results.append((n, round(m95, 3),
                    round(confidence(n, m95), 3)))


print("\nPROGRESSIVE MARGIN SEARCH")
print(pd.DataFrame(
    steps,
    columns=["Fractions", "Step", "Margin (mm)", "Confidence (%)"]
).to_string(index=False))

print("\n95% CONFIDENCE RESULTS")
print(pd.DataFrame(
    results,
    columns=["Fractions", "Margin (mm)", "Confidence (%)"]
).to_string(index=False))

print("\nCONFIDENCE BY MARGIN")
print(pd.DataFrame(
    [(n, m, round(confidence(n, m), 2))
     for n in (3, 30) for m in range(7)],
    columns=["Fractions", "Margin (mm)", "Confidence (%)"]
).to_string(index=False))


PROGRESSIVE MARGIN SEARCH
 Fractions  Step  Margin (mm)  Confidence (%)
         3     1       5.0000          96.988
         3     2       2.5000          86.000
         3     3       3.7500          93.157
         3     4       4.3750          95.366
         3     5       4.0625          94.364
         3     6       4.2188          94.891
         3     7       4.2969          95.119
         3     8       4.2578          95.008
         3     9       4.2383          94.950
         3    10       4.2480          94.976
        30     1       5.0000          99.990
        30     2       2.5000          98.182
        30     3       1.2500          87.622
        30     4       1.8750          94.921
        30     5       2.1875          96.958
        30     6       2.0312          96.036
        30     7       1.9531          95.503
        30     8       1.9141          95.224
        30     9       1.8945          95.072
        30    10       1.8848          94.999

95% CO